# TuRBO / Trust Region End-to-End Workflow

TuRBOはsurrogate modelやacquisition functionそのものではなく、**candidate search geometryを局所化するstateful strategy**です。

```text
completed data → surrogate → acquisition
                              ↓
incumbent → TuRBO state → local trust region → candidate
                                               ↓
                                            evaluate
                                               ↓
                    append data ← update_state(values, candidates)
```

このNotebookではpublic/original input space上でTuRBO-1 loopを実行します。

## 1. Imports and reproducibility

In [ ]:
import torch
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.fit import fit_gpytorch_mll

from robotorchan.models import SingleTaskGP
from robotorchan.optim import TuRBOState, TuRBOStrategy

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

## 2. Problem and initial observations

2次元のmaximize問題を使います。TuRBOのcenter、bounds、candidateはすべてpublic input coordinatesです。

In [ ]:
def objective(X: torch.Tensor) -> torch.Tensor:
    optimum = torch.tensor([0.72, 0.28])
    return -((X - optimum) ** 2).sum(dim=-1, keepdim=True)


dim = 2
bounds = torch.stack([torch.zeros(dim), torch.ones(dim)])
train_X = torch.rand(10, dim)
train_Y = objective(train_X)
best_index = train_Y.squeeze(-1).argmax()
initial_center = train_X[best_index].clone()
initial_best = float(train_Y[best_index].item())

print("initial center:", initial_center)
print("initial best:", initial_best)

## 3. TuRBO state and local geometry

`length`はglobal bounds幅に対する相対的なtrust-region sizeです。strategyはobjectiveを評価せず、local search regionとstate transitionを所有します。

In [ ]:
state = TuRBOState(
    dim=dim,
    batch_size=1,
    best_value=initial_best,
)
strategy = TuRBOStrategy(
    bounds,
    center=initial_center,
    state=state,
    num_restarts=3,
    raw_samples=64,
    seed=11,
)
local_bounds = strategy.trust_region_bounds()

assert local_bounds.shape == bounds.shape
assert torch.all(local_bounds >= bounds[0])
assert torch.all(local_bounds <= bounds[1])
print("trust-region length:", strategy.state.length)
print("local bounds:\n", local_bounds)

## 4. One TuRBO acquisition step

surrogate fittingとacquisition constructionは呼び出し側の責務です。TuRBOは同じacquisitionを現在のlocal bounds内で最適化します。

In [ ]:
model = SingleTaskGP(train_X, train_Y)
fit_gpytorch_mll(model.make_mll())
model.eval()
acquisition = qLogExpectedImprovement(
    model=model,
    best_f=train_Y.max(),
)
result = strategy.optimize(acquisition, q=1)
candidate = result.candidates.detach()
candidate_bounds = result.metadata["trust_region_bounds"]

assert candidate.shape == torch.Size([1, dim])
assert torch.all(candidate >= candidate_bounds[0])
assert torch.all(candidate <= candidate_bounds[1])
print("candidate:", candidate)
print("candidate-generation:", result.metadata["candidate_generation"])

## 5. Evaluation and state transition

評価完了後にのみ`update_state`を呼びます。改善candidateならincumbent centerも更新されます。

In [ ]:
new_Y = objective(candidate)
old_center = strategy.center.clone()
old_best = strategy.state.best_value
strategy.update_state(new_Y, candidates=candidate)

train_X = torch.cat([train_X, candidate], dim=0)
train_Y = torch.cat([train_Y, new_Y], dim=0)

print("observed value:", new_Y.item())
print("best value:", old_best, "->", strategy.state.best_value)
print("center:", old_center, "->", strategy.center)
print("success counter:", strategy.state.success_counter)
print("failure counter:", strategy.state.failure_counter)

## 6. End-to-end TuRBO-1 loop

各iterationでmodelをrefitし、acquisitionを再構築し、現在のtrust region内でcandidateを生成します。

In [ ]:
history = []
for iteration in range(4):
    model = SingleTaskGP(train_X, train_Y)
    fit_gpytorch_mll(model.make_mll())
    model.eval()
    acquisition = qLogExpectedImprovement(
        model=model,
        best_f=train_Y.max(),
    )
    result = strategy.optimize(acquisition, q=1)
    new_X = result.candidates.detach()
    new_Y = objective(new_X)
    strategy.update_state(new_Y, candidates=new_X)
    train_X = torch.cat([train_X, new_X], dim=0)
    train_Y = torch.cat([train_Y, new_Y], dim=0)
    history.append(
        (
            iteration,
            strategy.state.length,
            strategy.state.best_value,
            strategy.state.success_counter,
            strategy.state.failure_counter,
        )
    )

for row in history:
    print(row)

## 7. Thompson sampling is an alternative candidate generator

TuRBOではgradient-based acquisition optimizationだけでなく、現在のlocal regionに有限candidate poolを作りposterior Thompson samplingする経路も利用できます。

In [ ]:
model = SingleTaskGP(train_X, train_Y)
fit_gpytorch_mll(model.make_mll())
model.eval()
ts_result = strategy.thompson_sample(
    model,
    q=1,
    n_candidates=128,
)
ts_candidate = ts_result.candidates
ts_bounds = ts_result.metadata["trust_region_bounds"]

assert ts_candidate.shape == torch.Size([1, dim])
assert torch.all(ts_candidate >= ts_bounds[0])
assert torch.all(ts_candidate <= ts_bounds[1])
assert ts_result.metadata["candidate_generation"] == "thompson"
print("Thompson candidate:", ts_candidate)

## 8. Trust-region state is not optimizer state

TuRBO stateが保持するのはincumbent、trust-region length、success/failure counters、restart情報です。AdamやL-BFGS等のnumerical optimizer stateとは別物です。

```text
TuRBO strategy = where to search
acquisition optimizer = how to optimize inside that region
surrogate model = what posterior to believe
acquisition = what candidate utility to optimize
```

## 9. Restart semantics

`length < length_min`で`restart_triggered=True`になった場合、strategyは暗黙restartしません。`strategy.restart()`を明示的に呼び、新しいglobal re-entry centerから再開します。global bestは保持され、restart center自体を未評価のbest incumbentとは扱いません。

## 10. High-dimensional and reduced models

TuRBOは既定でpublic/original input spaceを局所化します。PCA/PLS/RandomProjection modelが内部latent representationを使っても、candidateとtrust-region geometryを暗黙にlatentへ移しません。

public dimensionと1対1対応するARD lengthscaleがある場合はdimension weightsへ反映できます。一方、latent GP lengthscaleをraw-space geometryへ暗黙逆写像しません。

## 11. Async, robust, mixed, and multi-objective boundaries

- **Async**: unresolved evaluationsはacquisitionの`X_pending`。TuRBO stateはcompleted resultsだけで更新します。
- **Fantasization**: model/acquisition responsibility。fantasy batchはTuRBO `batch_size`ではありません。
- **Robust BO**: input perturbation scenario axisはtrust-region dimensionへ追加しません。必要なら`state_values`にrobust utilityを渡します。
- **Mixed**: continuous dimensionsだけをlengthで局所化し、integer/categorical合法値集合へ人工的な数値距離を導入しません。
- **Multi-objective**: qLogEHVI/qLogNEHVI等はlocal regionで最適化できますが、TuRBO-1 state updateにはcandidateごとのscalar state utilityが必要です。
- **Candidate constraints**: trust regionは既知input constraintsの代替ではありません。両方を同時に満たす必要があります。

## 12. Related documentation

- `docs/optimization/turbo.md`
- `docs/optimization/high_dimensional_search.md`
- `docs/theory/optimization/06_trust_region.md`
- `31_acquisition_optimization.ipynb`
- `33_acquisition_initialization.ipynb`

TuRBOをモデルやoptimizerの別名として扱わず、**stateful local search geometry**として組み込むことがcross-layer contractの中心です。